# 41c · Pretrain ConvNeXtV2 trên PadChest — sửa **độ mạnh**, rồi đo lại **đa dạng**

## Vì sao notebook này tồn tại

`41a` đo được hai thứ ngược nhau:

```
corr voi RAD-DINO   0.47-0.52   so voi MOC same-backbone 0.8236   ->  DA DANG DAT rat thoang
mAP probe           0.1492      so voi RAD-DINO 0.2042            ->  QUA YEU
ensemble muc probe  nano+raddino 0.2006 vs raddino 0.2042 = -0.0035   ->  AM
```

Trọng số đều kéo tụt thành viên mạnh. **Nút chặn là độ mạnh, không phải đa dạng.**

PadChest sửa đúng chỗ đó và **sạch với CXR-LT 2024**: Hospital San Juan (Alicante) vs Beth Israel
(Boston) — không chung bệnh nhân, không chung ảnh, nên **không rò rỉ ảnh test**. Khác hẳn
`torchxrayvision -mimic_*`, thứ đã thấy ảnh test của ta **kèm nhãn**.

## Nhưng nó làm tăng đúng thứ ta sợ — nên `C4` là **cổng**, không phải phụ lục

Kéo CNN về miền CXR là kéo nó **về gần** RAD-DINO ⇒ **`corr` sẽ TĂNG**. `0.52` là ước lượng trên đặc
trưng ImageNet, **lạc quan**. Sau pretrain phải **đo lại**, không được mang số cũ sang dùng.

```
corr sau pretrain <  0.8236  ->  van con da dang -> di tiep 41b (fine-tune + ensemble)
corr sau pretrain >= 0.8236  ->  da dang BIEN MAT -> DONG huong ensemble CNN, du mAP co tang
```

## Ràng buộc thật là **I/O**, không phải GPU

```
964 GB nen / 83 MB/s  ~=  3.2 gio  chi de DOC mot lan
/content khong chua noi 964 GB  ->  chep TUNG zip -> giai nen + resize -> XOA -> zip tiep theo
KHONG mo zip truc tiep tren mount Drive: `zipfile` seek toi cuoi file 20-30 GB va FUSE
tra ve doc THIEU -> BadZipFile. Da dinh, va do la ly do C2 chep truoc khi mo.
```

Pretrain trên GPU chỉ `~2 h`. Khâu đắt là `C2`.

## C0 · Config

In [ ]:
import os, re, json, math, time, shutil, zipfile, tarfile, io as _io, collections
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    import timm
except ImportError:
    import subprocess, sys as _s
    subprocess.run([_s.executable, '-m', 'pip', 'install', '-q', 'timm']); import timm

try:
    from google.colab import drive; drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
    PC_ROOT  = '/content/drive/MyDrive/CXR-LT 2026'
except Exception:
    DATA_DIR = './planB_out'; PC_ROOT = './CXR-LT 2026'
PZ      = os.path.join(PC_ROOT, 'PadChest_Full')
INV_DIR = os.path.join(PC_ROOT, '_inventory')          # do 54 sinh ra
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RES41C  = os.path.join(OUT_DIR, 'stage4_nb41c')
RES41A  = os.path.join(OUT_DIR, 'stage4_nb41a')
RES49   = os.path.join(OUT_DIR, 'stage4_nb49')          # nguon MOC same-backbone
def ensure_res():
    '''Drive FUSE danh roi thu muc RONG — goi TRUOC moi cum ghi (43/38c/38d/41a da dinh).'''
    os.makedirs(RES41C, exist_ok=True); return RES41C
ensure_res(); json.dump({'notebook': '41c'}, open(os.path.join(RES41C, '_run_info.json'), 'w'))

MODEL   = 'convnextv2_nano.fcmae_ft_in22k_in1k'   # 41a: mAP cao nhat (0.1492), re nhat, corr 0.5193
LOC_PC  = '/content/padchest_518'                 # anh PadChest da resize
LOC_ZIP = '/content/_zip'                         # zip tam, XOA sau moi file
LOC_CXR = '/content/cxrlt_518'                    # anh CXR-LT (tu 45/53) — cho C4
SIZE, BATCH, ACCUM = 518, 256, 1        # HIEU DUNG van 256 — giu nguyen de so duoc voi cau hinh cu.
                                        # `convnextv2_nano` @518 chi ~13 GMAC; tren GPU 100 GB thi
                                        # DOC ANH la nut that, khong phai GPU. Batch 64 bo doi GPU.
                                        # GPU 16-40 GB thi tra ve `64, 4`.
# E=5 do LAT THEO DOI PadChest quyet dinh, khong phai doan: luot E=8 cho
#   ep1 .2563  ep2 .3080  ep3 .3525  ep4 .3697  ep5 .3837  ep6 .3696  ep7 .3629  ep8 .3600
# tuc dinh o ep5 roi troi xuong -0.0237 trong khi loss van giam (0.0340 -> 0.0190) = overfit.
# Cung hinh dang `04b` da do tren RAD-DINO. LAY CHECKPOINT ep5 cua luot E=8 KHONG tuong duong:
# cosine trai tren E, nen luot E=5 ha LR ve ~0 dung luc ket thuc con checkpoint giua chung thi khong.
EPOCHS, LR, WD, WARMUP = 5, 3e-4, 0.05, 0.05      # pretrain: LR nho hon vi fine-tune tu ImageNet
SEED, SEEDS = 42, (42, 43, 44)
PROBE_E, PROBE_LR, PROBE_WD = 60, 3e-3, 1e-4      # y het 49 C2 / 41a C2
NW = min(16, (os.cpu_count() or 2))     # batch 256 anh 518px -> 8 worker giai ma JPEG khong kip
HOLD = 0.05                                       # lat theo doi cua PadChest (theo BENH NHAN)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
# C0-C2 (stage data) la CPU + I/O thuan — chay duoc tren CPU runtime, khong ton quota GPU.
# Chi C3 tro di moi can GPU, va C3 tu chan lay.
STAGE_ONLY = not torch.cuda.is_available()
assert dev == 'cuda' or STAGE_ONLY, 'KHONG thay GPU'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
for p in (PZ, os.path.join(PC_ROOT, 'CXRLT_2026_training.csv'),
          os.path.join(PC_ROOT, 'Removal.csv'), os.path.join(PC_ROOT, 'Train_CLASS.txt')):
    assert os.path.exists(p), f'khong thay {p}'
if STAGE_ONLY:
    print(f'CPU runtime -> chi chay C0, C1, C2 (stage data). DUNG chay C3. | model {MODEL}')
else:
    print(f'thiet bi: {torch.cuda.get_device_name(0)} | AMP {AMP} | model {MODEL}')
print(f'PadChest: {PZ}\nghi vao : {RES41C}')
print(f'batch {BATCH} x accum {ACCUM} = hieu dung {BATCH*ACCUM} | {NW} worker')

## C1 · Chọn ảnh — `training − Removal`, đối chiếu kiểm kê của `54`

`Removal.csv` do ban tổ chức cung cấp liệt kê ảnh **dành cho dev/test của 2026** và **phải loại khỏi
mọi thủ tục huấn luyện**. Trừ nó xong thì tập này sạch với **cả** CXR-LT 2024 (PadChest ngoài
MIMIC-CXR) **và** 2026.

In [ ]:
T = pd.read_csv(os.path.join(PC_ROOT, 'CXRLT_2026_training.csv'), low_memory=False)
R = pd.read_csv(os.path.join(PC_ROOT, 'Removal.csv'), low_memory=False)
CL = [l.strip() for l in open(os.path.join(PC_ROOT, 'Train_CLASS.txt'), encoding='utf-8') if l.strip()]
K = next((c for c in T.columns if re.search(r'imageid|image_id|filename|^path$|^image$', str(c), re.I)),
         T.columns[0])
RK = next((c for c in R.columns if str(c).lower() == str(K).lower()), R.columns[0])
base = lambda s: os.path.basename(str(s)).strip()
T['_b'] = T[K].map(base); R['_b'] = R[RK].map(base)
print(f'training {len(T):,} | removal {len(R):,} | {len(CL)} lop | cot khoa {K}')

# --- KHONG con buoc KIEM KE ---
# Ban cu doc `padchest_inventory.parquet` cua `54` (hoac `*.unzip-l.txt`) de biet anh nao nam o zip
# nao. `54` chua bao gio chay duoc, va khi kiem ke RONG thi moi thu phia sau bao "thieu 100% anh"
# — mot chan doan SAI cho mot van de HOAN TOAN KHAC. C2 da chep tung zip ve dia va mo no roi, nen
# no TU biet ben trong co gi; khong can nguon su that thu hai de lech nhau.
_st = lambda x: os.path.splitext(str(x))[0]
T['_b'] = T._b.map(_st); R['_b'] = R._b.map(_st)      # ten KHONG DUOI — CSV ghi .png, ta luu .jpg
print('\n--- COHORT (kiem truoc khi tin bat cu con so nao) ---')
print(f'khoa CSV `{K}` 2 gia tri : {T[K].head(2).tolist()}')
print(f'  sau chuan hoa          : {T._b.head(2).tolist()}')
print(f'khoa Removal `{RK}`      : {R._b.head(2).tolist()}')
#  liet ke anh loai khoi PadChest DAY DU, con  la cohort DA
# loai roi -> giao BANG 0 la DUNG THIET KE. Phep tru o duoi chi la luoi an toan. KHONG dat cong o
# day: khong co nguong nao doc duoc tu con so nay. Cong THAT nam o cuoi C2 (>=95% cohort stage duoc)
# va dau C3 — chung do dung thu can do: anh co that tren dia hay khong.
_hit = len(set(T._b) & set(R._b))
print(f'Removal giao training    : {_hit:,} / {len(R):,}  '
      f'({_hit/max(len(R),1):.1%}; 0% la binh thuong)')

T['_keep'] = ~T._b.isin(set(R._b))
U = T[T._keep].reset_index(drop=True)
print(f'\n{"":<26}{"so anh":>10} {"ty le":>8}')
print(f'{"training goc":<26}{len(T):>10,}')
print(f'{"sau khi tru Removal":<26}{len(U):>10,} {len(U)/len(T):>8.1%}')
print('  (bao nhieu anh THUC SU stage duoc do C2 chot — xem cong 0.95 o cuoi C2)')

YPC = np.zeros((len(U), len(CL)), 'float32')
for j, c in enumerate(CL):
    col = next((x for x in U.columns if str(x).strip().lower() == c.strip().lower()), None)
    if col is not None: YPC[:, j] = pd.to_numeric(U[col], errors='coerce').fillna(0).gt(0.5)
print(f'nhan {YPC.shape} | prevalence TB {YPC.mean():.3%} | '
      f'{int((YPC.sum(0) > 0).sum())}/{len(CL)} lop co ca duong')
assert (YPC.sum(0) > 0).sum() >= len(CL) - 2, 'qua nhieu lop khong co ca duong -> sai cot nhan'

# lat theo doi: chia theo BENH NHAN neu co cot, khong thi theo anh
_pid = next((c for c in U.columns if re.search(r'patient', str(c), re.I)), None)
_rs = np.random.default_rng(SEED)
if _pid is not None:
    _u = U[_pid].astype(str).to_numpy(); _uq = np.unique(_u)
    _h = set(_rs.choice(_uq, size=max(1, int(round(HOLD*len(_uq)))), replace=False))
    HO = np.isin(_u, list(_h))
    print(f'lat theo doi chia theo `{_pid}`: {HO.sum():,} anh / {len(_h):,} benh nhan')
else:
    HO = _rs.random(len(U)) < HOLD
    print(f'!! khong thay cot benh nhan -> chia theo ANH ({HO.sum():,}). Lat theo doi CO THE lac quan.')
TRp = ~HO
SEL_N = set(U._b)          # ten (khong duoi) ma C2 se tim trong tung zip
print(f'\ncohort cho C2: {len(SEL_N):,} ten anh')

## C2 · Stage — chép **từng** zip về `/content`, resize 518, đẩy **kho bền vững** lên Drive

`/content` bị xoá mỗi lần session khởi động lại, và một lượt stage tốn ~4.5 giờ. Nên mỗi zip xong
là **đóng gói ngay thành tar và đẩy lên Drive**: session chết giữa chừng chỉ mất **một** zip, và
mọi notebook PadChest sau này hydrate lại trong vài phút thay vì decode lại từ đầu.

**Ba dạng hư hỏng của kho zip này, cả ba đều phải chịu được** (không dạng nào tự báo):

```
1. ca zip khong mo duoc         vd `50.zip` — 14 MB, dung 1 anh, nam tron trong tien to 429
2. zf.read(n) nem loi           member nam trong vung bi tien to 429 de len
3. zf.read(n) OK nhung byte cut -> PIL chet `image file is truncated` GIUA vong lap nhieu gio
```

Dạng 3 là dạng đã giết một lượt chạy: `_conv` chạy trong `ThreadPoolExecutor` nên **một** ảnh hỏng
ném lỗi là dừng cả buổi. Nay mọi lỗi giải mã bị nuốt vào `BAD` và đếm, kèm **cổng 2%** để phân biệt
*vài ảnh lẻ* (đúng như đã đo, ~0.017%) với *zip hỏng cả vùng* (phải tải lại).

**PadChest lưu PNG 16-bit.** `Image.convert('L')` trên mode `I;16` **không** phải phép chia 256 — nó
cắt byte, cho ra ảnh sọc nhiễu **mà không ném lỗi**. Phải tự cửa sổ theo phân vị 0.5/99.5.

**Không bao giờ xoá tay `content_cache` của DriveFS.** Cache đọc không tự dọn (mỗi zip 22 GB đọc vào
là 22 GB nằm lại, hết 226 GB đĩa ở zip thứ ~8), nhưng DriveFS dùng **chính thư mục đó** làm chỗ chứa
ghi đang chờ upload — xoá tay là huỷ ghi. Đã làm hỏng `_staged.json` và hai tar đầu. Cách đúng là
`flush_and_unmount` (đẩy ghi lên trước) rồi mount lại.

**Nguồn sự thật của resume là các `.tar` có thật trong kho**, không phải `_staged.json` — json chỉ
còn ghi các zip không mở được (chúng không sinh tar).

In [ ]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

LOC_IMG = LOC_PC                                         # anh 518 da resize (C0)
READY   = os.path.join(OUT_DIR, 'padchest_518_ready')    # KHO BEN VUNG — /content bi xoa moi restart
CHUNK   = 512                                            # anh/lo; dinh RAM ~2.5 GB
MIN_FREE = 80e9                                          # duoi muc nay thi giai phong dia
for _d in (LOC_IMG, LOC_ZIP): os.makedirs(_d, exist_ok=True)
def _ready():
    os.makedirs(READY, exist_ok=True); return READY      # Drive FUSE danh roi thu muc RONG
P_SD = os.path.join(_ready(), '_staged.json')
_tarof = lambda z: os.path.join(READY, f'pc518_{os.path.splitext(z)[0]}.tar')

assert os.path.isdir(OUT_DIR) and os.listdir(OUT_DIR), (
    f'{OUT_DIR} khong doc duoc -> mount Drive HONG. Chay lai cell mount (flush_and_unmount + mount) '
    'hoac Restart session truoc. Mount hong khien `os.path.exists` tra ve False, va C2 se tuong la '
    'chua stage gi roi lam lai tu dau 4.5 tieng.')
_pt = os.path.join(_ready(), '_perm_test')
open(_pt, 'w').write('ok')
assert open(_pt).read() == 'ok', f'{READY}: khong GHI duoc (tai khoan chi co quyen Viewer?)'
os.remove(_pt)

def _reclaim(force=False):
    """Cache doc cua Drive FUSE khong tu don: moi zip 22 GB doc vao la 22 GB nam lai.
       KHONG duoc xoa tay `content_cache` — DriveFS dung chinh no lam cho chua ghi DANG CHO
       upload, xoa la HUY ghi (da lam hong `_staged.json` va hai tar dau).
       `flush_and_unmount` DAY ghi len Drive TRUOC roi moi giai phong."""
    fr = shutil.disk_usage('/').free
    if not force and fr > MIN_FREE: return 0.0
    try:
        from google.colab import drive
    except ImportError:
        return 0.0                                       # khong phai Colab -> khong co cache FUSE
    drive.flush_and_unmount(); drive.mount('/content/drive')
    d = (shutil.disk_usage('/').free - fr) / 1e9
    print(f'     giai phong dia: +{d:.0f} GB | con {shutil.disk_usage("/").free/1e9:.0f} GB',
          flush=True)
    return d

# ---------- ghep cap: chuan hoa CA HAI ben giong nhau, bat ke C1 co cat duoi hay khong ----------
assert 'SEL_N' in globals(), 'thieu `SEL_N` -> chay lai C1 truoc'
_norm = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
SEL = SEL_N
ZIPS = sorted(f for f in os.listdir(PZ) if f.endswith('.zip'))   # KHONG can biet truoc anh nao o
assert ZIPS, f'{PZ} khong co file .zip nao'                      # zip nao — mo ra la biet
print(f'cohort: {len(SEL):,} anh | {len(ZIPS)} zip tren dia')
assert len(SEL) > 100_000, f'chi chon duoc {len(SEL):,} anh -> xem lai khoi COHORT o C1'

# ---------- hydrate: TAR la nguon su that, khong phai json ----------
for z in ZIPS:
    tp = _tarof(z)
    if not os.path.exists(tp): continue
    tmp = os.path.join('/content', os.path.basename(tp))
    try:
        shutil.copy(tp, tmp)                             # copy TRUOC — khong mo tar tren mount FUSE
        assert os.path.getsize(tmp) == os.path.getsize(tp), 'copy NGAN tu Drive'
        with tarfile.open(tmp) as tar: tar.extractall(LOC_IMG, filter='data')
    except Exception as e:
        print(f'  {os.path.basename(tp)} HONG ({type(e).__name__}) -> xoa, se stage lai {z}')
        os.remove(tp)
    if os.path.exists(tmp): os.remove(tmp)
done = {z for z in ZIPS if os.path.exists(_tarof(z))}    # tar co that == da xong
try:
    done |= {z for z in json.load(open(P_SD)) if z in ZIPS}   # zip KHONG MO DUOC khong sinh tar
except Exception:
    print('_staged.json rong/hong -> suy `done` tu cac tar co san (json chi la phu tro)')
_n = len([f for f in os.listdir(LOC_IMG) if f.endswith('.jpg')])
print(f'hydrate tu kho: {_n:,} anh | {len(done)}/{len(ZIPS)} zip da xong')

# ---------- chuyen doi mot anh ----------
BAD = []
def _conv(pl):
    """Moi loi giai ma BI NUOT vao BAD. `_conv` chay trong ThreadPoolExecutor nen mot anh hong
       nem loi la GIET ca lan chay nhieu gio (dung `image file is truncated`).
       KHONG bat LOAD_TRUNCATED_IMAGES: anh cut giai ma ra nua duoi mau xam — them NHIEU vao
       corpus pretrain chu khong them du lieu."""
    n, blob, made = pl
    dst = os.path.join(LOC_IMG, _norm(n) + '.jpg')
    if os.path.exists(dst): made.append(dst); return
    try:
        im = Image.open(_io.BytesIO(blob)); im.load()
        a = np.asarray(im)
        if a.ndim == 3: a = a.mean(-1)
        if a.dtype != np.uint8:
            # PadChest luu PNG 16-BIT (mode 'I;16'). `convert('L')` cua PIL tren mode nay KHONG
            # phai phep chia 256 — no CAT BYTE, cho ra anh soc nhieu ma KHONG nem loi.
            a = a.astype('float32')
            lo, hi = np.percentile(a, (0.5, 99.5))       # cua so ban le, chuan cho PadChest
            a = np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1) * 255
        im = Image.fromarray(a.astype('uint8')).resize((SIZE, SIZE), Image.BICUBIC)
    except Exception as e:
        BAD.append((n, type(e).__name__)); return
    tmp = dst + '.tmp'                                   # ghi tam roi doi ten: `os.replace` nguyen
    im.save(tmp, 'JPEG', quality=95); os.replace(tmp, dst)   # tu => `exists(dst)` == "ghi XONG",
    made.append(dst)                                         # khong con nghia "da bat dau ghi"

# ---------- vong lap theo tung zip ----------
for _f in os.listdir(LOC_ZIP): os.remove(os.path.join(LOC_ZIP, _f))   # don xac lan chay truoc
t0 = time.time()
for z in ZIPS:
    if z in done: continue
    src = os.path.join(PZ, z); tmp = os.path.join(LOC_ZIP, z); n0 = time.time()
    _reclaim()                                           # flush + remount NEU dia sap het
    _free = shutil.disk_usage('/').free
    assert _free > 3 * os.path.getsize(src), (
        f'con {_free/1e9:.0f} GB trong, {z} nang {os.path.getsize(src)/1e9:.0f} GB. Can gap 3 '
        '(ban sao + cache FUSE + du phong). Het dia giua chung thi `shutil.copy` de lai file CUT '
        'ma KHONG phai luc nao cung nem loi -> zip that se bi ghi nham la "hong".')
    try:
        shutil.copy(src, tmp)                            # CHEP TRUOC — khong mo zip tren mount
    except OSError as _e:
        raise RuntimeError(
            f'{z}: doc that bai giua chung ({type(_e).__name__}). Metadata doc duoc nhung noi dung '
            'thi khong -> mount Drive hong (hoac dang bi throttle 429 vi doc song song). Remount '
            'roi chay lai C2; `done` suy tu tar nen khong mat tien do.') from _e
    if os.path.getsize(tmp) != os.path.getsize(src):
        print(f'  {z:<10} COPY NGAN {os.path.getsize(tmp):,}/{os.path.getsize(src):,} '
              '-> chay lai cell de thu lai', flush=True)
        os.remove(tmp); continue                         # KHONG danh dau done -> lan sau lam lai
    try:
        zipfile.ZipFile(tmp).close()
    except Exception as _e:
        # HU HONG 1: ca zip khong mo duoc (vd `50.zip` — 14 MB, dung 1 anh, nam tron trong tien to 429)
        print(f'  {z:<10} KHONG MO DUOC ({type(_e).__name__}) -> BO QUA ca zip', flush=True)
        os.remove(tmp); done.add(z); json.dump(sorted(done), open(P_SD, 'w')); continue

    made = []; _skip = 0; _b0 = len(BAD)
    with zipfile.ZipFile(tmp) as zf:
        mem = [n for n in zf.namelist() if _norm(n) in SEL]   # kham pha TAI CHO, khong kiem ke
        with ThreadPoolExecutor(8) as pool:
            for k in range(0, len(mem), CHUNK):          # doc THEO LO: mot zip PadChest ~3,000 anh
                jobs = []                                # PNG 16-bit ~5 MB. Gom het la ~15 GB.
                for n in mem[k:k+CHUNK]:
                    try:
                        jobs.append((n, zf.read(n), made))
                    except Exception:
                        # HU HONG 2: 26/52 zip co tien to `429 Too Many Requests` cua nginx
                        # (2.5-8.9 MB) do cong cu tai noi trang loi vao file. `zipfile` tu bu offset
                        # qua EOCD nen phan lon member doc binh thuong; chi vai anh nam vung bi de.
                        _skip += 1
                list(pool.map(_conv, jobs)); del jobs
    os.remove(tmp)
    _nb = len(BAD) - _b0                                 # HU HONG 3: read() OK nhung BYTE CUT

    # --- dong goi NGAY len Drive: session chet giua chung chi mat MOT zip ---
    if made:
        tp = os.path.join('/content', os.path.basename(_tarof(z)))
        with tarfile.open(tp, 'w') as tar:
            for p in made: tar.add(p, arcname=os.path.basename(p))
        shutil.copy(tp, _tarof(z)); os.remove(tp)
        assert os.path.getsize(_tarof(z)) > 0, f'{_tarof(z)} ghi ra 0 byte'

    _n = len([f for f in os.listdir(LOC_IMG) if f.endswith('.jpg')])
    print(f'  {z:<10} {len(made):>7,}/{len(mem):,} anh | {os.path.getsize(src)/1e9:5.1f} GB | '
          f'{(time.time()-n0)/60:5.1f} ph | tong {_n:,} | da troi {(time.time()-t0)/60:.0f} ph'
          + (f' | BO QUA {_skip} member (429)' if _skip else '')
          + (f' | HONG {_nb} anh' if _nb else ''), flush=True)
    _lost = _skip + _nb                              # dat TRUOC `done.add` -> zip hong KHONG
    assert _lost <= 20 or _lost < 0.02 * max(len(mem), 1), (
        f'{z}: {_lost}/{len(mem)} anh mat ({_lost/max(len(mem),1):.1%}). Vuot CA HAI nguong '
        '(>20 anh VA >2%) -> tien to 429 da de ca mot VUNG chu khong mot entry; tai lai zip nay. '
        "Ti le THUAN khong dung: `50.zip` chi co 1 anh nen mat 1 la 100%, ma 1 anh thi khong bao "
        'gio la bang chung cua vung bi de.')
    done.add(z); json.dump(sorted(done), open(P_SD, 'w'))

_reclaim(force=True)                                     # day not phan ghi cuoi len Drive
_n = len([f for f in os.listdir(LOC_IMG) if f.endswith('.jpg')])
print(f'\nstage xong: {_n:,}/{len(SEL):,} anh ({_n/len(SEL):.2%}) | hong tong {len(BAD)} '
      f'| kho ben vung: {READY}')
assert _n > 0.95 * len(SEL), f'chi stage duoc {_n/len(SEL):.1%} -> DUNG pretrain voi tap nay'

## C3 · Pretrain `ConvNeXtV2-nano` trên 30 nhãn PadChest

In [ ]:
# `U` cua C1 la cohort THEO CSV. Anh thuc su stage duoc do C2 quyet dinh (zip hong, member 429,
# 3 zip 51/52/53 von khong ton tai trong PadChest). Loc o day — dung `os.path.exists`, khong dung
# mot ban kiem ke thu hai co the lech.
_ex = U._b.map(lambda b: os.path.exists(os.path.join(LOC_PC, f'{b}.jpg'))).to_numpy()
print(f'anh co tren dia: {_ex.sum():,}/{len(U):,} ({_ex.mean():.1%})')
assert _ex.mean() > 0.95, (f'chi {_ex.mean():.1%} cohort co anh -> chay lai C2 cho xong '
                           '(hoac kiem lai zip nao chua tai duoc)')
U = U[_ex].reset_index(drop=True); YPC = YPC[_ex]; HO = HO[_ex]; TRp = ~HO
print(f'pretrain tren {int(TRp.sum()):,} anh | lat theo doi {int(HO.sum()):,}')

assert dev == 'cuda', ('cell nay can GPU — doi runtime sang GPU roi chay lai C0, C1, C2, C3. '
                       'C2 se hydrate tu kho tren Drive (vai phut), khong stage lai.')
from torchvision import transforms as T2
class ASL(nn.Module):
    '''Verbatim nb11/12/13/15/35/37/38*/44*/45/49/52*/53. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()

def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o

_m0 = timm.create_model(MODEL, pretrained=True, num_classes=0, in_chans=1)
_cfg = timm.data.resolve_model_data_config(_m0)
MU, SD = float(np.mean(_cfg['mean'])), float(np.mean(_cfg['std']))
NF = _m0.num_features; del _m0
tf_tr = T2.Compose([T2.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                         interpolation=T2.InterpolationMode.BICUBIC),
                    T2.RandomRotation(7)])
NAMES = [os.path.splitext(b)[0] for b in U._b]

class DSP(Dataset):
    def __init__(s, idx, train): s.idx, s.train = np.asarray(idx), train
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        i = int(s.idx[k])
        im = Image.open(os.path.join(LOC_PC, NAMES[i] + '.jpg'))
        if s.train: im = tf_tr(im)
        x = torch.from_numpy(np.asarray(im, dtype='uint8').copy()).unsqueeze(0).float().div_(255)
        return (x - MU) / SD, torch.from_numpy(YPC[i])

P_BB = os.path.join(RES41C, 'convnextv2_padchest.pt')
if os.path.exists(P_BB):
    print(f'nap lai {P_BB}')
else:
    torch.manual_seed(SEED)
    net = timm.create_model(MODEL, pretrained=True, num_classes=len(CL), in_chans=1).to(dev)
    crit = ASL(gn=4, gp=0).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WD)
    dl = DataLoader(DSP(np.where(TRp)[0], True), batch_size=BATCH, shuffle=True, num_workers=NW,
                    pin_memory=True, drop_last=True, persistent_workers=NW > 0)
    dv = DataLoader(DSP(np.where(HO)[0], False), batch_size=BATCH, shuffle=False, num_workers=NW,
                    pin_memory=True)
    steps = math.ceil(int(TRp.sum()) / (BATCH*ACCUM)) * EPOCHS; warm = int(WARMUP*steps)
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s/max(warm,1) if s < warm else
                                            0.5*(1+math.cos(math.pi*(s-warm)/max(steps-warm,1))))
    for ep in range(EPOCHS):
        net.train(); t0, tot, nb = time.time(), 0.0, 0
        opt.zero_grad(set_to_none=True)
        for j, (x, y) in enumerate(tqdm(dl, desc=f'pretrain ep{ep+1}/{EPOCHS}', leave=False)):
            with torch.autocast('cuda', dtype=AMP):
                l = crit(net(x.to(dev, non_blocking=True)).float(), y.to(dev))
            (l / ACCUM).backward(); tot += float(l.detach()); nb += 1
            if (j+1) % ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
                opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        net.eval(); ps = []
        with torch.no_grad():
            for x, _ in dv:
                with torch.autocast('cuda', dtype=AMP):
                    ps.append(torch.sigmoid(net(x.to(dev)).float()).cpu().numpy())
        ap = ap_cols(YPC[HO], np.concatenate(ps)); mp = float(np.nanmean(ap))
        print(f'ep{ep+1:>2}/{EPOCHS} loss {tot/max(nb,1):.4f} | PadChest mAP {mp:.4f} | '
              f'{nb*BATCH/max(time.time()-t0,1e-9):.0f} img/s | {(time.time()-t0)/60:.1f} ph',
              flush=True)
        # Ghi LOCAL truoc roi copy len Drive, va DOC LAI kiem. `torch.save` thang vao mount tra ve
        # thanh cong trong khi ghi con nam hang doi; hong o do thi IM LANG. Da mat mot luot 80 phut
        # y het the: `ensure_res()` tao lai thu muc, nhung Drive cho phep HAI thu muc TRUNG TEN nen
        # file roi vao ban sao ma FUSE khong hien.
        ensure_res()
        P_BB_LOC = '/content/convnextv2_padchest.pt'      # GIU LAI lam ban du phong trong session
        torch.save(dict(sd=net.state_dict(), ep=ep+1, mAP=mp, model=MODEL,
                        classes=CL, n_train=int(TRp.sum())), P_BB_LOC)
        shutil.copy(P_BB_LOC, P_BB)
        assert os.path.exists(P_BB) and os.path.getsize(P_BB) > 1e6, (
            f'{P_BB} khong hien ra sau khi luu -> Drive nuot ghi. DUNG NGAY o epoch nay, dung chay '
            'tiep roi phat hien khi da het E epoch.')
    _pv = float(np.nanmean(YPC[HO].mean(0)))
    assert mp > 2.5 * _pv, (f'PadChest mAP {mp:.4f} chi {mp/_pv:.1f}x prevalence {_pv:.4f} -> '
                            'pretrain KHONG hoc duoc gi. Dung dung checkpoint nay.')
    print(f'cong 3: PadChest mAP {mp:.4f} = {mp/_pv:.1f}x san prevalence {_pv:.4f} OK')
    del net; torch.cuda.empty_cache()

## C3-bis · Stage ảnh **CXR-LT** 518 — thứ `C4` cần, khác hẳn PadChest của `C2`

`C2` stage **PadChest** (miền pretrain). `C4` đo `corr` với RAD-DINO trên **CXR-LT** — dữ liệu đích,
377,110 ảnh, nguồn khác. Trước đây phải chạy `C2` của `45`/`53` để có chúng; cell này làm tại chỗ.

Đọc từ shard `cxr_1022_letterbox_cxrlt/lb_*.tar` (~49 GB) rồi hạ về 518. Cùng kỷ luật đã rút ra ở
`C2`: đọc **bulk** cả tar vào RAM (lặp trên mount FUSE sẽ treo), kiểm **độ dài** bản đọc (FUSE trả
về đọc ngắn mà không báo lỗi), ghi **nguyên tử** (`tmp` rồi `os.replace`, để `exists` nghĩa là *ghi
xong* chứ không phải *đã bắt đầu ghi*), và **không bao giờ** xoá tay `content_cache`.

Vì đằng nào cũng phải trả tiền đọc 49 GB, nó đóng gói luôn thành **kho bền vững**
`cxrlt_518_ready/` trên Drive (~12–15 GB). Lần sau — kể cả cho `45`/`53`/`56` — chỉ còn copy + bung,
vài phút thay vì 40.

In [ ]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

if 'ORDER' not in dir():
    ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')
READY_X   = os.path.join(OUT_DIR, 'cxrlt_518_ready')     # kho ben vung, dung chung cho 45/53/56
MIN_FREE_X, N_TH = 60e9, 16
os.makedirs(LOC_CXR, exist_ok=True)
def _readyx():
    """Goi TRUOC MOI lan ghi, khong chi mot lan dau: Drive FUSE danh roi thu muc, va mot duong dan
       tinh tu dau vong lap se tro vao cho khong con ton tai (43/38c/38d/41a deu da dinh)."""
    os.makedirs(READY_X, exist_ok=True); return READY_X
_tarof_x = lambda t: os.path.join(READY_X, 'x518_' + os.path.splitext(t)[0] + '.tar')

def _reclaim_x():
    """Cache doc cua Drive FUSE khong tu don. KHONG xoa tay `content_cache` — DriveFS dung chinh
       no lam cho chua ghi DANG CHO upload (da lam hong `_staged.json` cua C2)."""
    if shutil.disk_usage('/').free > MIN_FREE_X: return
    try:
        from google.colab import drive
    except ImportError:
        return
    drive.flush_and_unmount(); drive.mount('/content/drive')
    print(f'     giai phong dia -> con {shutil.disk_usage("/").free/1e9:.0f} GB', flush=True)

_nx = len([f for f in os.listdir(LOC_CXR) if f.endswith('.jpg')])
print(f'anh CXR-LT dang co: {_nx:,} / {len(ORDER):,}')

# ---------- 1) hydrate tu kho neu da co ----------
if _nx < 0.999 * len(ORDER) and os.path.isdir(READY_X):
    for f in sorted(x for x in os.listdir(READY_X) if x.endswith('.tar')):
        src = os.path.join(READY_X, f); tmp = os.path.join('/content', f)
        try:
            shutil.copy(src, tmp)                        # copy TRUOC — khong mo tar tren mount
            assert os.path.getsize(tmp) == os.path.getsize(src), 'copy NGAN tu Drive'
            with tarfile.open(tmp) as tar: tar.extractall(LOC_CXR, filter='data')
        except Exception as e:
            print(f'  {f} HONG ({type(e).__name__}) -> xoa, se stage lai phan do')
            os.remove(src)
        if os.path.exists(tmp): os.remove(tmp)
    _nx = len([f for f in os.listdir(LOC_CXR) if f.endswith('.jpg')])
    print(f'hydrate tu kho: {_nx:,} / {len(ORDER):,}')

# ---------- 2) chua du thi stage tu shard goc ----------
if _nx < 0.999 * len(ORDER):
    assert os.path.isdir(SHARD_DIR), f'khong thay {SHARD_DIR}'
    TARS = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar') and f.startswith('lb_'))
    assert TARS, f'{SHARD_DIR} khong co lb_*.tar'
    # Nguon su that la TAR CO THAT trong kho; json chi la phu tro. Mot file json mat/hong khong
    # duoc phep xoa sach tien do — bai hoc tu `_staged.json` cua C2.
    donex = {t for t in TARS if os.path.exists(_tarof_x(t))}
    try:
        donex |= {t for t in json.load(open(os.path.join(_readyx(), '_staged.json'))) if t in TARS}
    except Exception:
        pass
    print(f'{len(TARS)} tar | da xong {len(donex)}')

    def _one(pl):
        nm, blob = pl
        dst = os.path.join(LOC_CXR, os.path.basename(nm))
        if os.path.exists(dst): return dst
        try:
            im = Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC)
        except Exception:
            return None                                  # mot anh hong KHONG duoc giet ca lan chay
        tmp = dst + '.tmp'
        im.save(tmp, 'JPEG', quality=95); os.replace(tmp, dst)   # ghi NGUYEN TU
        return dst

    t0 = time.time()
    for t in TARS:
        if t in donex: continue
        _reclaim_x()
        tp = os.path.join(SHARD_DIR, t)
        with open(tp, 'rb') as f: buf = f.read()         # doc BULK — lap tren mount FUSE se TREO
        assert len(buf) == os.path.getsize(tp), (
            f'{t}: doc {len(buf):,}/{os.path.getsize(tp):,} byte -> doc NGAN tren mount. '
            'Chay lai cell; KHONG danh dau tar nay la xong.')
        with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
            jobs = [(m.name, tar.extractfile(m).read())
                    for m in tar.getmembers() if m.name.endswith('.jpg')]
        del buf
        with ThreadPoolExecutor(N_TH) as pool: made = [p for p in pool.map(_one, jobs) if p]
        _bad = len(jobs) - len(made); del jobs

        if made:                                         # dong goi NGAY: session chet chi mat 1 tar
            _tp = os.path.join('/content', os.path.basename(_tarof_x(t)))
            with tarfile.open(_tp, 'w') as tar:
                for p in made: tar.add(p, arcname=os.path.basename(p))
            shutil.copy(_tp, os.path.join(_readyx(), os.path.basename(_tp))); os.remove(_tp)
            assert os.path.getsize(_tarof_x(t)) > 0, 'tar kho ghi ra 0 byte'
        _nx = len([f for f in os.listdir(LOC_CXR) if f.endswith('.jpg')])
        print(f'  {t:<22} {len(made):>6,} anh | tong {_nx:>7,} | {(time.time()-t0)/60:5.1f} ph'
              + (f' | HONG {_bad}' if _bad else ''), flush=True)
        donex.add(t)
        json.dump(sorted(donex), open(os.path.join(_readyx(), '_staged.json'), 'w'))

_nx = len([f for f in os.listdir(LOC_CXR) if f.endswith('.jpg')])
print(f'\nCXR-LT san sang: {_nx:,}/{len(ORDER):,} ({_nx/len(ORDER):.2%}) | kho: {READY_X}')
assert _nx >= 0.999 * len(ORDER), (
    f'moi co {_nx:,}/{len(ORDER):,}. C4 trich dac trung theo CHI SO HANG cua `ORDER`, thieu anh la '
    'hang do thanh rac ma khong bao loi -> phai du 99.9% moi chay tiep.')

## C4 · **CỔNG** — đo lại `corr` với RAD-DINO trên CXR-LT

Đây là lý do notebook này tồn tại. Pretrain kéo CNN về miền CXR ⇒ `corr` **sẽ tăng**. Nếu nó vượt
`MOC same-backbone` của `49` thì đa dạng biến mất và hướng ensemble CNN đóng, **dù mAP có tăng**.

Trích đặc trưng trên **ảnh CXR-LT `lb518`** (đã stage sẵn ở `/content/cxrlt_518`), probe tuyến tính
**y hệt** `49 C2`/`41a C2`, rồi so với `p_lb518_s42.npy` của `49`.

In [ ]:
from scipy.stats import rankdata
INVX = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INVX['labels_task1']; C40 = len(LAB40)
TIER = {t: np.array([LAB40.index(l) for l in INVX['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
YX = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
        .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())[LAB40].to_numpy('float32')
SP = ORDER.split.to_numpy(); SB = ORDER.subject_id.to_numpy()
TRx, VAx = SP == 'train', SP == 'val'
_r2 = np.random.default_rng(SEED); _s2 = np.unique(SB[TRx])
_sel2 = _r2.choice(_s2, size=int(round(0.15*len(_s2))), replace=False)
SLx = TRx & np.isin(SB, _sel2); TFx = TRx & ~SLx
assert os.path.isdir(LOC_CXR), (
    f'khong thay {LOC_CXR}. C4 can anh CXR-LT (KHAC PadChest cua C2/C3) de do `corr` voi RAD-DINO '
    '-> chay C2 cua 45/53 de stage ~49 GB, khoang 40-60 phut.')
_ncx = len([f for f in os.listdir(LOC_CXR) if f.endswith('.jpg')])
assert _ncx >= 0.999 * len(ORDER), (
    f'{LOC_CXR} chi co {_ncx:,}/{len(ORDER):,} anh -> stage CHUA XONG. Kiem THU MUC TON TAI thoi '
    'la khong du: `DSX` se chet giua vong lap sau khi da chay mot luc, va checkpoint dac trung '
    'se ghi do dang.')
print(f'anh CXR-LT: {_ncx:,}/{len(ORDER):,}')

class DSX(Dataset):
    def __init__(s, n): s.n = n
    def __len__(s): return s.n
    def __getitem__(s, i):
        im = Image.open(os.path.join(LOC_CXR, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        x = torch.from_numpy(np.asarray(im, dtype='uint8').copy()).unsqueeze(0).float().div_(255)
        return (x - MU) / SD, i

# Memmap ghi TRUC TIEP len mount Drive la cach chac chan nhat de mat 965 MB sau 30 phut:
# FUSE khong bao dam thu tu ghi, `flush()` chi day xuong hang doi, va thu muc co the bi danh roi
# giua chung (da dinh 6 lan trong du an). Ghi LOCAL — nhanh hon nhieu va dang tin — roi copy len
# Drive MOT lan o cuoi, co doc lai kiem.
P_FE_LOC = '/content/feat_cxrlt.npy'
P_FE     = os.path.join(RES41C, 'feat_cxrlt.npy')
P_PG     = '/content/_feat_prog.json'          # chot resume di kem file local
if not os.path.exists(P_FE_LOC) and os.path.exists(P_FE):
    shutil.copy(P_FE, P_FE_LOC)                # lan chay truoc da xong -> keo ve dung lai
    if os.path.exists(os.path.join(RES41C, '_feat_prog.json')):
        shutil.copy(os.path.join(RES41C, '_feat_prog.json'), P_PG)
n0 = json.load(open(P_PG))['n'] if os.path.exists(P_PG) else 0
if n0 < len(ORDER):
    net = timm.create_model(MODEL, pretrained=False, num_classes=0, in_chans=1)
    _src_bb = P_BB if os.path.exists(P_BB) else '/content/convnextv2_padchest.pt'
    assert os.path.exists(_src_bb), (
        f'khong thay trong so o CA {P_BB} lan /content -> chay lai C3. '
        'Neu C3 vua chay xong trong session nay ma Drive khong co, kiem thu muc trung ten tren '
        'Drive web truoc khi ton 50 phut chay lai.')
    print(f'nap backbone tu {_src_bb}')
    _ck = torch.load(_src_bb, map_location='cpu', weights_only=False)
    _miss = net.load_state_dict({k: v for k, v in _ck['sd'].items()
                                 if not k.startswith('head.')}, strict=False)
    assert not [k for k in _miss.missing_keys if not k.startswith('head.')], \
        f'thieu khoa khi nap backbone: {_miss.missing_keys[:5]}'
    net = net.to(dev).eval()
    for p in net.parameters(): p.requires_grad_(False)
    FM = np.lib.format.open_memmap(P_FE_LOC, mode=('r+' if os.path.exists(P_FE_LOC) else 'w+'),
                                   dtype='float32', shape=(len(ORDER), NF))
    dl = DataLoader(DSX(len(ORDER)), batch_size=BATCH, shuffle=False, num_workers=NW,
                    pin_memory=True)
    t0, seen = time.time(), 0
    with torch.no_grad():
        for x, ii in tqdm(dl, desc='trich CXR-LT'):
            if int(ii[-1]) < n0: seen = int(ii[-1]) + 1; continue
            with torch.autocast('cuda', dtype=AMP):
                f = net(x.to(dev, non_blocking=True))
            FM[ii.numpy()] = f.float().cpu().numpy(); seen = int(ii[-1]) + 1
            if seen % (BATCH*50) < BATCH:
                FM.flush(); json.dump({'n': seen}, open(P_PG, 'w'))
    FM.flush(); json.dump({'n': len(ORDER)}, open(P_PG, 'w'))
    del FM, net; torch.cuda.empty_cache()
    ensure_res(); shutil.copy(P_FE_LOC, P_FE)          # MOT lan, o cuoi
    shutil.copy(P_PG, os.path.join(RES41C, '_feat_prog.json'))
    assert os.path.getsize(P_FE) == os.path.getsize(P_FE_LOC), (
        f'{P_FE} copy len Drive KHONG du byte ({os.path.getsize(P_FE):,} vs '
        f'{os.path.getsize(P_FE_LOC):,}). Ban local van con o {P_FE_LOC}, khong mat gi.')
    print(f'trich xong {(time.time()-t0)/60:.1f} ph | {os.path.getsize(P_FE)/1e6:.0f} MB len Drive')

X = np.array(np.load(P_FE_LOC, mmap_mode='r'))
assert np.isfinite(X).all() and np.abs(X).sum(1).min() > 0, 'dac trung co hang toan 0'
def fit(Xf, seed):
    '''EPOCH CUOI, cosine trai tren PROBE_E, KHONG early-stop (35 §0.1). Chuan hoa bang LAT KHOP.'''
    torch.manual_seed(seed)
    mu, sd = Xf[TFx].mean(0, keepdims=True), Xf[TFx].std(0, keepdims=True) + 1e-6
    z = lambda m: torch.from_numpy(((Xf[m]-mu)/sd).astype('float32')).to(dev)
    xt, xv = z(TFx), z(VAx); yt = torch.from_numpy(YX[TFx]).to(dev)
    m = nn.Linear(xt.shape[1], C40).to(dev); cr = ASL(gn=4, gp=0).to(dev)
    op = torch.optim.AdamW(m.parameters(), lr=PROBE_LR, weight_decay=PROBE_WD)
    sc = torch.optim.lr_scheduler.CosineAnnealingLR(op, PROBE_E)
    bs = int(min(4096, max(256, len(xt)//64)))
    for _ in range(PROBE_E):
        m.train(); pm = torch.randperm(len(xt), device=dev)
        for i in range(0, len(xt), bs):
            k = pm[i:i+bs]; op.zero_grad(set_to_none=True); cr(m(xt[k]), yt[k]).backward(); op.step()
        sc.step()
    m.eval()
    with torch.no_grad(): return torch.sigmoid(m(xv)).cpu().numpy()

PS = {}
for s in SEEDS:
    f = os.path.join(RES41C, f'p_pc_s{s}.npy')
    PS[s] = np.load(f) if os.path.exists(f) else fit(X, s)
    if not os.path.exists(f):
        ensure_res(); np.save(f, PS[s])
        assert os.path.exists(f), f'{f} khong hien ra sau khi luu -> Drive nuot ghi'
    print(f'probe s{s}  mAP {float(np.nanmean(ap_cols(YX[VAx], PS[s]))):.4f}')
MAP_PC = float(np.mean([np.nanmean(ap_cols(YX[VAx], PS[s])) for s in SEEDS]))

def corr(A, B):
    ra, rb = rankdata(A, axis=0), rankdata(B, axis=0)
    ra = (ra - ra.mean(0)) / (ra.std(0) + 1e-12); rb = (rb - rb.mean(0)) / (rb.std(0) + 1e-12)
    return float(np.mean((ra * rb).mean(0)))
_g49 = lambda a, s: os.path.join(RES49, f'p_{a}_s{s}.npy')
A49 = [a for a in ('lb518','nosq518','crop518','lb672') if os.path.exists(_g49(a, SEEDS[0]))]
assert 'lb518' in A49, f'khong thay {_g49("lb518", SEEDS[0])} -> chay 49 truoc'
P49 = {a: np.load(_g49(a, SEEDS[0])) for a in A49}
MOC = min(corr(P49[a], P49[b]) for i, a in enumerate(A49) for b in A49[i+1:])
RD = P49['lb518']; MAP_RD = float(np.nanmean(ap_cols(YX[VAx], RD)))
CORR_PC = corr(PS[SEEDS[0]], RD)
_a41 = os.path.join(RES41A, f'p_{MODEL.split(".")[0]}_s{SEEDS[0]}.npy')
CORR_IN = corr(np.load(_a41), RD) if os.path.exists(_a41) else float('nan')
MAP_IN = float(np.nanmean(ap_cols(YX[VAx], np.load(_a41)))) if os.path.exists(_a41) else float('nan')

print(f'\n{"":<24}{"mAP":>8} {"corr voi RAD-DINO":>19}')
print(f'{"ImageNet (41a)":<24}{MAP_IN:>8.4f} {CORR_IN:>19.4f}')
print(f'{"+ PadChest (41c)":<24}{MAP_PC:>8.4f} {CORR_PC:>19.4f}')
print(f'{"RAD-DINO":<24}{MAP_RD:>8.4f} {"—":>19}')
print(f'\nMOC same-backbone (49) = {MOC:.4f}')
print('\n=== CONG ===')
print(f'  1  mAP tang       {MAP_PC-MAP_IN:+.4f}  ({"DAT" if MAP_PC > MAP_IN else "KHONG"})')
print(f'  2  corr < {MOC:.4f}   {CORR_PC:.4f}  ({"DAT" if CORR_PC < MOC else "KHONG"})')
print(f'     corr tang do pretrain: {CORR_PC-CORR_IN:+.4f}')
if CORR_PC >= MOC:
    print(f'\n  !! corr {CORR_PC:.4f} >= MOC {MOC:.4f} -> DA DANG BIEN MAT.')
    print('     Keo CNN ve mien CXR da keo no ve gan RAD-DINO. DONG huong ensemble CNN,')
    print('     du mAP co tang. Ghi ket qua AM nay vao bao cao.')
elif MAP_PC <= MAP_IN:
    print('\n  !! mAP KHONG tang -> pretrain khong sua duoc do manh. Xem lai C3 (E, LR).')
else:
    print(f'\n  -> CA HAI cong DAT. Di tiep 41b: fine-tune tren CXR-LT roi ensemble.')
    print(f'     Cong cua 41b: CNN >= 0.85 x RAD-DINO fine-tune (38a 0.2412 -> >= 0.2050)')
    print(f'     VA corr DO LAI SAU fine-tune < {MOC:.4f} VA ensemble - manh nhat >= +0.005')

## C5 · Ensemble miễn phí ở mức probe + ghi `REPORT_STAGE4_NB41C.md`

In [ ]:
rk = lambda P: rankdata(P, axis=0) / len(P)
_p = float(np.nanmean(ap_cols(YX[VAx], (PS[SEEDS[0]] + RD) / 2)))
_r = float(np.nanmean(ap_cols(YX[VAx], (rk(PS[SEEDS[0]]) + rk(RD)) / 2)))
_best = max(MAP_PC, MAP_RD)
print(f'{"to hop":<34}{"prob":>8}{"rank":>8}{"vs tot nhat":>13}')
print(f'{"padchest-cnn + raddino":<34}{_p:>8.4f}{_r:>8.4f}{max(_p,_r)-_best:>+13.4f}')
if not np.isnan(MAP_IN):
    _pi = float(np.nanmean(ap_cols(YX[VAx], (np.load(_a41) + RD) / 2)))
    print(f'{"(41a) imagenet-cnn + raddino":<34}{_pi:>8.4f}{"—":>8}{_pi-max(MAP_IN,MAP_RD):>+13.4f}')
print(f'\n  tot nhat don le {_best:.4f} | Trong so DEU — chon trong so tren val roi bao cao tren '
      'val la dung bay §13.')
print('  Day la ensemble o muc PROBE (dac trung dong cung), KHONG phai ket qua cuoi.')

L = []; w = L.append
w('# REPORT · `nb41c` — pretrain ConvNeXtV2 trên PadChest'); w('')
w(f'`{MODEL}` · `{int(TRp.sum()):,}` ảnh PadChest (`CXRLT_2026_training` − `Removal`) · '
  f'`{len(CL)}` nhãn · `asl40` · `E={EPOCHS}` · probe tuyến tính trên CXR-LT val '
  f'`{int(VAx.sum()):,}` ảnh, seeds {list(SEEDS)}.'); w('')
w('| | mAP | `corr` với RAD-DINO |'); w('|---|---|---|')
w(f'| ImageNet (`41a`) | {MAP_IN:.4f} | {CORR_IN:.4f} |')
w(f'| **+ PadChest (`41c`)** | **{MAP_PC:.4f}** | **{CORR_PC:.4f}** |')
w(f'| RAD-DINO | {MAP_RD:.4f} | — |')
w(''); w(f'```'); w(f'MOC same-backbone (49) = {MOC:.4f}')
w(f'mAP  {MAP_IN:.4f} -> {MAP_PC:.4f}   ({MAP_PC-MAP_IN:+.4f})')
w(f'corr {CORR_IN:.4f} -> {CORR_PC:.4f}   ({CORR_PC-CORR_IN:+.4f})'); w('```'); w('')
w('**Vì sao PadChest sạch với CXR-LT 2024.** PadChest là Hospital San Juan (Alicante), MIMIC-CXR là '
  'Beth Israel (Boston) — không chung bệnh nhân, không chung ảnh. Split test của CXR-LT 2024 là tập '
  'con của MIMIC-CXR nên pretrain trên PadChest **không** làm lộ ảnh test. `Removal.csv` của ban tổ '
  'chức loại sẵn ảnh dành cho dev/test của **2026**.'); w('')
w('## Những gì kết quả này KHÔNG trả lời'); w('')
w('- **`corr` sẽ còn tăng nữa sau fine-tune trên CXR-LT.** Fine-tune trên nhãn đích kéo **mọi** mô '
  f'hình về cùng cấu trúc nhãn. `{CORR_PC:.4f}` vẫn là ước lượng **lạc quan** cho `41b`; phải đo lại.')
w('- **Probe trên đặc trưng đông cứng**, không phải fine-tune. `04b` đo LP `0.3191` → FT `0.4201` '
  '(`+0.101`) trên cohort cũ ⇒ thứ hạng có thể đổi.')
w('- **Ràng buộc giấy phép.** Nhãn CXR-LT 2026 *"may not be shared under any circumstances"* và phải '
  'xoá sau `04/11/2026`; luật thi ghi *"may only train your models on the 30 labels"*. Việc dùng '
  'chúng để pretrain cho một **giải khác** là câu hỏi cho ban tổ chức.')
TXT = '\n'.join(L); ensure_res()
open(os.path.join(RES41C, 'REPORT_STAGE4_NB41C.md'), 'w', encoding='utf-8').write(TXT)
print('\n' + TXT)